# Explainability (Grad-CAM & SHAP)
Diving into the 'Why' behind our model's predictions. using Grad-CAM to visualize CNN attention, and SHAP to understand which nutritional features heavily penalized or boosted our health scores.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shap
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.models import efficientnet_b0

from src.food_classifier import FoodEfficientNet
from src.health_scorer import HealthScoreEngine

# Ensure figures directory exists
os.makedirs('../reports/figures', exist_ok=True)

## 1. Grad-CAM on Food Classifier
Where is the EfficientNet model focusing when classifying the image?

In [ ]:
# Initialize model (mocking loaded weights)
model = FoodEfficientNet(num_classes=20, freeze_backbone=False)
model.eval()

# Target the final Convolutional layer in EfficientNet features
target_layers = [model.model.features[-1]]

# Instantiate the CAM object
cam = GradCAM(model=model, target_layers=target_layers)

# Mock input image tensor representing an ingested photo (1 batch, 3 channels, 224x224)
input_tensor = torch.randn(1, 3, 224, 224)

# Compute CAM
grayscale_cam = cam(input_tensor=input_tensor, targets=None)
grayscale_cam = grayscale_cam[0, :]

# Construct Visualization on a mock RGB base image
rgb_img = np.random.rand(224, 224, 3)
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(6, 6))
plt.imshow(visualization)
plt.title('Grad-CAM Model Attention')
plt.axis('off')
plt.savefig('../reports/figures/gradcam_sample.png', bbox_inches='tight')
plt.show()

## 2. SHAP on Health Score Engine
Which macronutrients drove the health score down (or up)?

In [ ]:
# Wrapper function to expose the engine's predict function into a SHAP compatible pipeline
def health_score_predict(data):
    engine = HealthScoreEngine()
    scores = []
    for row in data:
        nut_dict = {
            'calories': row[0],
            'protein_g': row[1],
            'fat_g': row[2],
            'carbs_g': row[3],
            'sugar_g': row[4],
            'sodium_mg': row[5]
        }
        scores.append(engine.calculate_score(nut_dict))
    return np.array(scores)

# Nutritional feature dimensions
features = ['calories', 'protein_g', 'fat_g', 'carbs_g', 'sugar_g', 'sodium_mg']

# Generate mock reference distribution (background data) for KernelExplainer
background_data = np.random.rand(50, 6) * [1000, 50, 50, 100, 20, 1500]
explainer = shap.KernelExplainer(health_score_predict, background_data)

# Explaining a single isolated meal instance (e.g., Unhealthy Fast Food)
test_meal = np.array([[1200, 45, 60, 110, 15, 1500]])
shap_values = explainer.shap_values(test_meal, print_obj=False)

plt.figure()
shap.summary_plot(shap_values, test_meal, feature_names=features, show=False)
plt.savefig('../reports/figures/shap_summary.png', bbox_inches='tight')
plt.show()